# PART 2. Model training and reconstruction

Learn across development sites using campaign context, simultaneous references, and ERA5. Fixed held-out validation episodes select the checkpoint; test sites are excluded from training and model selection. Reconstruction can accept any target, including a test site for final evaluation. Shared model and inference definitions live in `wind_pipeline.py`; training-only helpers stay here.

Measured wind is assumed to be at 3 m throughout the study. Notebook 1 retains measurement height in its saved metadata table. Height is output metadata only; `prepare_training_data` accepts coordinate/elevation metadata as predictors, and notebook 2 uses that same retained column order for training, validation, and reconstruction. Run cells in order. Run All prepares the data, trains the model, saves the best validation checkpoint, and runs the smoke check. Full-period reconstruction remains a commented example.

This follow-up keeps the existing architecture and site split while averaging four episodes per update, reducing the learning rate and direction weight, and increasing dropout. Campaign-adaptation architecture changes require coordinated updates to the shared model and inference code.


In [ ]:
"""------------------------ ADJUSTABLE PARAMETER CONTROL PANEL ------------------------"""
# FILE PATHS
WIND_PATH = "analyses_output_files/wind_pipeline/observations/wind.parquet"     # Cleaned wind
ERA5_PATH = "analyses_output_files/wind_pipeline/observations/era5.parquet"     # Cleaned ERA5
METADATA_PATH = "analyses_output_files/wind_pipeline/metadata/metadata.parquet" # Site metadata

# DATA PREPARATION
DEVELOPMENT_SITES = ("az06", "az05", "az22", "az26")    # Rotating training sites
VALIDATION_SITES = ("az19",)        # Model selection sites
TEST_SITES = ("az15",)              # Final evaluation sites
START_LOCAL = "2005-01-01"          # First requested date
END_LOCAL = "2019-12-31"            # Last requested date
FIT_END_LOCAL = "2015-12-31"        # Last training date
N_REFERENCES = 2                    # References per episode
CAMPAIGN_MONTHS = (3,)              # Allowed campaign lengths
SITE_METADATA = (                   # Coordinate/elevation predictors only
    "latitude_deg",
    "longitude_deg",
    "ground_elevation_m",
)

# EPISODE SAMPLING
N_CONTEXT = 256                     # Maximum context hours, C
N_QUERY = 256                       # Maximum training query hours, Q
MIN_CONTEXT_HOURS = 24              # Minimum usable campaign hours
MIN_CONTEXT_FRACTION = 0.5          # Minimum usable campaign fraction
MIN_QUERY_HOURS = 1                 # Minimum usable query hours
MAX_SAMPLING_ATTEMPTS = 100         # Attempts before sampling fails

# MODEL ARCHITECTURE
D_MODEL = 64                        # Embedding width, D
N_HEADS = 4                         # Attention heads, h; must divide D
D_FF = 128                          # Feed-forward hidden width
L_ENC = 2                           # Context encoder layers
L_DEC = 1                           # Query cross-attention layers
DROPOUT = 0.2                       # Training dropout probability
ACTIVATION = "gelu"                 # Feed-forward activation
LAYER_NORM_EPS = 1e-5               # Layer normalization stability

# OPTIMIZER
OPTIMIZER = "AdamW"                 # Weight update algorithm
LEARNING_RATE = 1e-4                # Update step size
ADAM_BETAS = (0.9, 0.999)           # Gradient averaging rates
ADAM_EPS = 1e-8                     # Optimizer numerical stability
WEIGHT_DECAY = 1e-4                 # Weight regularization strength
GRAD_CLIP_NORM = 1.0                # Maximum total gradient norm

# LEARNING-RATE SCHEDULE
LR_REDUCTION_FACTOR = 0.5          # Multiply LR when validation stalls
LR_SCHEDULER_PATIENCE = 2          # Unsuccessful checks before reducing LR
MIN_LEARNING_RATE = 1e-6           # Scheduler floor

# LOSS
SPEED_LOSS_WEIGHT = 1.0             # Standardized speed MSE weight
DIRECTION_LOSS_WEIGHT = 0.25         # Mean sin/cos MSE weight

# TRAINING AND VALIDATION
RANDOM_SEED = 42                    # Reproducible random choices
DEVICE = "auto"                     # CUDA if available, otherwise CPU
MAX_TRAINING_STEPS = 10000          # Maximum optimizer updates
EPISODES_PER_UPDATE = 4             # Episodes averaged per update
VALIDATION_INTERVAL = 250           # Updates between validation checks
N_VALIDATION_EPISODES = 50          # Fixed episodes per validation
VALIDATION_SEED = 123               # Reproducible validation episodes
EARLY_STOPPING_PATIENCE = 10        # Checks without improvement
EARLY_STOPPING_MIN_DELTA = 1e-4     # Minimum validation loss improvement
LOG_INTERVAL = 50                   # Updates between progress reports
RECONSTRUCTION_QUERY_CHUNK = 1024   # Query hours processed together
RECONSTRUCTION_CONTEXT_MODE = "full" # "trained" count or "full" campaign
CHECKPOINT_PATH = "analyses_output_files/model/best_context_model.pt"           # Best model file
VALIDATION_CAMPAIGN_START_DATES = ("2011-10-01", "2013-01-01", "2014-07-01", "2015-04-01")    # Local starts
LOCAL_TIMEZONE = "America/Phoenix"  # Observation clock
CALM_SPEED_MPS = 0.0                # Undefined direction at calm speeds
DIRECTION_VECTOR_EPS = 1e-8         # Numerically zero direction vector


## One-time preparation - notes step 0

$$z=\frac{x-\mu_{\mathrm{train}}}{\sigma_{\mathrm{train}}}.$$

`prepare_training_data` already performs this transformation, pooling development training observations through the fitting cutoff. Speed, ERA5 components, and retained metadata are standardized. Direction and local midpoint clock sin/cos stay unchanged. Missing values are zero with separate Boolean masks; entirely unobserved metadata is excluded everywhere. Preparation runs once, outside training.


In [ ]:
from pathlib import Path
from time import perf_counter
from tempfile import TemporaryDirectory
from itertools import combinations
import numpy as np
import pandas as pd
import torch
from torch import nn
from wind_pipeline import (
    prepare_training_data, sample_training_episode, ERA_COLUMNS,
    WindContextModel, plain_normalization, restore_normalization, load_checkpoint,
    load_aligned_sites, assemble_predictors, campaign_indices,
    assemble_reconstruction_inputs, reconstruct_wind)

np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
if LOCAL_TIMEZONE != "America/Phoenix":
    raise ValueError("The existing preparation/sampler contract requires America/Phoenix.")
prepared_data = prepare_training_data(
    WIND_PATH, ERA5_PATH, METADATA_PATH,
    development_sites=DEVELOPMENT_SITES, validation_sites=VALIDATION_SITES,
    test_sites=TEST_SITES, fit_end_local=FIT_END_LOCAL,
    n_references=N_REFERENCES, campaign_months=CAMPAIGN_MONTHS,
    start=START_LOCAL, end=END_LOCAL, metadata_columns=SITE_METADATA)
FEATURE_NAMES = prepared_data['feature_names']
TARGET_NAMES = prepared_data['target_names']
F = len(FEATURE_NAMES)
if any(name.endswith('measurement_height_m') for name in FEATURE_NAMES):
    raise ValueError("Measurement height must remain output metadata, not a model predictor.")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(device) if device.type == "cuda" else "none")
print("Predictors:", F, "| targets:", TARGET_NAMES)
print("Site roles:", {k: prepared_data[k] for k in ('development_sites', 'validation_sites', 'test_sites')})
print("Prepared UTC endpoints:", pd.to_datetime(prepared_data['times_utc'][[0, -1]], utc=True).tolist())
print("Retained predictor metadata:", prepared_data['metadata_columns'])
print("Excluded unobserved metadata:", prepared_data['excluded_unobserved_metadata'])


## Episode tensors - notes step 1

Sample fresh episodes with the supplied RNG. Tensor shapes are `[1,C,F]`, `[1,C,3]`, `[1,Q,F]`, and `[1,Q,3]`, with matching masks. C/Q may be smaller than the maxima. IDs, timestamps, and campaign details stay separate.


In [ ]:
def episode_to_tensors(episode, device):
    tensors = {}
    for prefix in ('context', 'query'):
        for kind in ('x', 'y'):
            key = f'{prefix}_{kind}'
            tensors[key] = torch.as_tensor(episode[key], dtype=torch.float32, device=device).unsqueeze(0)
            tensors[key + '_mask'] = torch.as_tensor(episode[key + '_mask'], dtype=torch.bool, device=device).unsqueeze(0)
    details = {k: v for k, v in episode.items() if k not in tensors}
    return tensors, details


def sample_episode_tensors(data, rng, device, n_context, n_query, *,
                           min_context_hours, min_context_fraction,
                           min_query_hours, max_attempts):
    episode = sample_training_episode(
        data, rng, n_context=n_context, n_query=n_query,
        min_context_hours=min_context_hours, min_context_fraction=min_context_fraction,
        min_query_hours=min_query_hours, max_attempts=max_attempts)
    return episode_to_tensors(episode, device)


sampling_settings = dict(n_context=N_CONTEXT, n_query=N_QUERY,
                         min_context_hours=MIN_CONTEXT_HOURS, min_context_fraction=MIN_CONTEXT_FRACTION,
                         min_query_hours=MIN_QUERY_HOURS, max_attempts=MAX_SAMPLING_ATTEMPTS)


## Context Model Architecture

**1. Input projections**, with observed masks cast to floats:
$$E_c=\operatorname{Concat}(X_c,M_{xc},Y_c,M_{yc})W_c+b_c,\quad
E_q=\operatorname{Concat}(X_q,M_{xq})W_q+b_q.$$

**2. Context self-attention**, repeated with separate parameters per encoder layer:
$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}(QK^T/\sqrt{d_k})V,
\quad d_k=D_{\mathrm{model}}/N_{\mathrm{heads}},$$
$$Z_c=\operatorname{LN}(E_c+\operatorname{Dropout}(\operatorname{MHA}(E_c,E_c,E_c))),$$
$$F_c=\operatorname{Linear}_2(\operatorname{Dropout}(\operatorname{GELU}(\operatorname{Linear}_1(Z_c)))),\quad
H_c=\operatorname{LN}(Z_c+\operatorname{Dropout}(F_c)).$$

**3. Query cross-attention**, repeated with separate parameters; all layers attend to the final encoded context:
$$Z_q=\operatorname{LN}(E_q+\operatorname{Dropout}(\operatorname{MHA}(E_q,H_c,H_c))),$$
$$F_q=\operatorname{Linear}_2(\operatorname{Dropout}(\operatorname{GELU}(\operatorname{Linear}_1(Z_q)))),\quad
H_q=\operatorname{LN}(Z_q+\operatorname{Dropout}(F_q)).$$

**4. Linear output**: $\widehat Y_q=H_qW_{\mathrm{head}}+b_{\mathrm{head}}$.

Every residual addition precedes LayerNorm. Feature masks enter projections, never attention padding masks. Clock features supply time information. Queries have no self-attention or positional embeddings; query labels enter only the loss. Outputs are unconstrained standardized speed, direction sine, and direction cosine.

Implementation is imported from `wind_pipeline.py`; the equations and model parameter names are unchanged.


In [ ]:
def predict_episode(model, tensors):
    return model(tensors['context_x'], tensors['context_x_mask'],
                 tensors['context_y'], tensors['context_y_mask'],
                 tensors['query_x'], tensors['query_x_mask'])


## Masked loss - notes step 3

$$L_s=\frac{\sum_i M_{si}(\widehat s_i-s_i)^2}{\sum_i M_{si}},$$
$$L_{\mathrm{dir}}=\frac{\sum_i[M_{\sin,i}(\widehat{\sin}_i-\sin_i)^2+
M_{\cos,i}(\widehat{\cos}_i-\cos_i)^2]}{\sum_i(M_{\sin,i}+M_{\cos,i})},$$
$$L_{\mathrm{episode}}=w_sL_s+w_{\mathrm{dir}}L_{\mathrm{dir}}.$$

Only observed label components contribute. Speed supervision is required; an entirely missing direction contributes zero.


In [ ]:
def masked_episode_loss(prediction, target, target_mask, speed_weight, direction_weight, *, return_components=False):
    speed_mask = target_mask[..., 0]
    if not speed_mask.any():
        raise ValueError("An episode requires usable speed supervision.")
    speed_loss = (prediction[..., 0][speed_mask] - target[..., 0][speed_mask]).square().mean()
    direction_mask = target_mask[..., 1:]
    # Select before subtraction: missing labels never enter the loss.
    errors = prediction[..., 1:][direction_mask] - target[..., 1:][direction_mask]
    direction_loss = errors.square().sum() / direction_mask.sum().clamp_min(1)
    total = speed_weight * speed_loss + direction_weight * direction_loss
    return (total, speed_loss, direction_loss) if return_components else total


## Held-out loading and fixed validation episodes

Apply the existing training statistics without fitting. Validation loads only validation observations, then reuses the cached development references. Reconstruction shares this loader, restricting target wind reads to the campaign and keeping reference predictors over the requested period.

Assembly follows the sampler's reference slots, target ERA5/metadata, then four local midpoint clock features. Validation campaigns and scoring queries must fit inside the cutoff-limited period. Coverage uses the existing essential-data rule before subsampling: target speed and all target ERA5 components, plus every reference's speed and all ERA5 components. Missing direction is allowed. Episodes are fixed once using the validation RNG and held on CPU.


In [ ]:
def build_validation_episodes(data, rng, validation_sites, development_sites, campaign_start_dates,
                              campaign_months, n_references, n_episodes, n_context, n_query,
                              min_context_hours, min_context_fraction, min_query_hours,
                              feature_names, target_names, metadata_columns, local_timezone):
    if n_episodes < 1 or not campaign_start_dates or not validation_sites:
        raise ValueError("Validation requires episodes, sites, and explicit campaign starts.")
    # Validate every configured campaign, even if a random draw would omit it.
    candidates = []
    for target in validation_sites:
        for date in campaign_start_dates:
            begin = pd.Timestamp(date).normalize().tz_localize(local_timezone)
            for months in campaign_months:
                finish = (begin + pd.DateOffset(months=months)).tz_convert('UTC')
                begin_utc = begin.tz_convert('UTC')
                viable = []
                # Only a few development sites; retain viable reference IDs, not histories.
                for refs in combinations(development_sites, n_references):
                    try:
                        ci, qi = campaign_indices(data, target, refs, begin_utc, finish,
                                                  min_context_hours, min_context_fraction)
                    except ValueError:
                        continue
                    if len(qi) >= min_query_hours:
                        viable.append(tuple(refs))
                if not viable:
                    raise ValueError(f"Unsuitable validation campaign: {target}, {date}, {months} months; check period and essential-data coverage.")
                candidates.append((target, begin_utc, finish, months, viable))
    episodes = []
    for _ in range(n_episodes):
        target, begin, finish, months, viable = candidates[int(rng.integers(len(candidates)))]
        refs = tuple(rng.permutation(viable[int(rng.integers(len(viable)))]).tolist())
        ci, qi = campaign_indices(data, target, refs, begin, finish, min_context_hours, min_context_fraction)
        usable_count = len(ci)
        ci = np.sort(rng.choice(ci, min(n_context, len(ci)), replace=False))
        qi = np.sort(rng.choice(qi, min(n_query, len(qi)), replace=False))
        cx, cm = assemble_predictors(data, target, refs, ci, feature_names, metadata_columns)
        qx, qm = assemble_predictors(data, target, refs, qi, feature_names, metadata_columns)
        tar = data['sites'][target]
        episodes.append(dict(context_x=cx, context_x_mask=cm, context_y=tar['wind'][ci].copy(),
                             context_y_mask=tar['wind_mask'][ci].copy(), query_x=qx, query_x_mask=qm,
                             query_y=tar['wind'][qi].copy(), query_y_mask=tar['wind_mask'][qi].copy(),
                             target_site_id=target, reference_site_ids=refs, campaign_months=months,
                             campaign_start_utc=begin, campaign_end_utc=finish,
                             context_times_utc=data['times_utc'][ci].copy(), query_times_utc=data['times_utc'][qi].copy(),
                             usable_context_hours=usable_count, feature_names=list(feature_names), target_names=list(target_names)))
    return episodes


validation_end_local = str((prepared_data['end_utc'].tz_convert(LOCAL_TIMEZONE)-pd.DateOffset(days=1)).date())
held_out_validation = load_aligned_sites(
    WIND_PATH, ERA5_PATH, METADATA_PATH, VALIDATION_SITES, START_LOCAL, validation_end_local,
    prepared_data['normalization'], prepared_data['metadata_columns'], LOCAL_TIMEZONE)
np.testing.assert_array_equal(held_out_validation['times_utc'], prepared_data['times_utc'])
validation_data = dict(held_out_validation)
validation_data['sites'] = {**prepared_data['sites'], **held_out_validation['sites']}
validation_episodes = build_validation_episodes(
    validation_data, np.random.default_rng(VALIDATION_SEED), VALIDATION_SITES, DEVELOPMENT_SITES,
    VALIDATION_CAMPAIGN_START_DATES, CAMPAIGN_MONTHS, N_REFERENCES, N_VALIDATION_EPISODES,
    N_CONTEXT, N_QUERY, MIN_CONTEXT_HOURS, MIN_CONTEXT_FRACTION, MIN_QUERY_HOURS,
    FEATURE_NAMES, TARGET_NAMES, prepared_data['metadata_columns'], LOCAL_TIMEZONE)
print("Fixed validation episodes:", len(validation_episodes), "| UTC period:",
      pd.to_datetime(validation_data['times_utc'][[0, -1]], utc=True).tolist())


## Initialize the model and optimizer - notes step 0

$$\theta_0=\operatorname{Initialize}(\mathrm{architecture},\mathrm{seed}),\qquad m_0=0,\quad v_0=0.$$

The model parameters are learned. AdamW's moments are optimizer state, initialized lazily at the first update. Model initialization follows the seed set during setup.


In [ ]:
architecture = dict(n_features=F, n_targets=len(TARGET_NAMES), d_model=D_MODEL, n_heads=N_HEADS,
                    d_ff=D_FF, l_enc=L_ENC, l_dec=L_DEC, dropout=DROPOUT,
                    activation=ACTIVATION, layer_norm_eps=LAYER_NORM_EPS)
model = WindContextModel(**architecture).to(device)
if OPTIMIZER != "AdamW":
    raise ValueError("This workflow uses AdamW.")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS,
                              eps=ADAM_EPS, weight_decay=WEIGHT_DECAY)
preprocessing = dict(
    local_timezone=LOCAL_TIMEZONE, start_local=START_LOCAL, end_local=END_LOCAL,
    fit_end_local=FIT_END_LOCAL, fit_end_utc=prepared_data['fit_end_utc'].isoformat(),
    training_end_utc=prepared_data['end_utc'].isoformat(),
    n_references=N_REFERENCES, campaign_months=list(CAMPAIGN_MONTHS),
    min_context_hours=MIN_CONTEXT_HOURS, min_context_fraction=MIN_CONTEXT_FRACTION,
    min_query_hours=MIN_QUERY_HOURS, n_context=N_CONTEXT, n_query=N_QUERY,
    validation_campaign_start_dates=list(VALIDATION_CAMPAIGN_START_DATES),
    validation_seed=VALIDATION_SEED, random_seed=RANDOM_SEED,
    timestamp_convention=prepared_data['timestamp_convention'],
    clock_convention='local hourly interval midpoints; calendar-year day-of-year fraction',
    direction_convention='sin/cos of degrees; direction valid only at positive observed speed',
    missing_convention='zero-filled values with True=observed Boolean masks',
    measurement_height_assumption='3 m throughout study; recorded in metadata, excluded from predictors',
    calm_speed_mps=CALM_SPEED_MPS, direction_vector_eps=DIRECTION_VECTOR_EPS,
    excluded_unobserved_metadata=list(prepared_data['excluded_unobserved_metadata']))
site_roles = {k: list(prepared_data[k]) for k in ('development_sites', 'validation_sites', 'test_sites')}
print("Parameters:", sum(p.numel() for p in model.parameters()), "| head width:", model.head_width)


## Checkpoints and validation

Every strictly lower validation loss saves a checkpoint; minimum delta controls only patience. Normalization is serialized as plain dictionaries and restored without fitting. Save only `state_dict` plus inference settings. The load example becomes usable after training writes a checkpoint.


In [ ]:
def save_checkpoint(path, model, architecture, feature_names, target_names, metadata_columns,
                    normalization, site_roles, preprocessing, best_step, validation_loss):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = dict(
        model_state_dict=model.state_dict(), architecture=dict(architecture),
        feature_names=list(feature_names), target_names=list(target_names),
        metadata_columns=list(metadata_columns), normalization=plain_normalization(normalization),
        site_roles=dict(site_roles), preprocessing=dict(preprocessing),
        model_identity=f'WindContextModel:step={best_step}:validation_loss={validation_loss:.8g}',
        best_step=int(best_step), validation_loss=float(validation_loss))
    torch.save(checkpoint, path)




def validate_model(model, episodes, device, speed_weight, direction_weight, *, return_components=False):
    was_training = model.training
    model.eval()
    losses, speed_losses, direction_losses = [], [], []
    try:
        with torch.no_grad():
            for episode in episodes:
                tensors, _ = episode_to_tensors(episode, device)
                loss, speed_loss, direction_loss = masked_episode_loss(
                    predict_episode(model, tensors), tensors['query_y'], tensors['query_y_mask'],
                    speed_weight, direction_weight, return_components=True)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Nonfinite validation loss.")
                losses.append(loss.item())
                speed_losses.append(speed_loss.item())
                direction_losses.append(direction_loss.item())
    finally:
        model.train(was_training)
    if not losses:
        raise ValueError("Validation requires fixed episodes.")
    total = float(np.mean(losses))
    if return_components:
        return total, float(np.mean(speed_losses)), float(np.mean(direction_losses))
    return total


# loaded_model, saved, saved_normalization = load_checkpoint(CHECKPOINT_PATH, device)


## Training - notes steps 1-8

1. Sample $B=\mathrm{EPISODES\_PER\_UPDATE}$ fresh episodes.
2. Predict $\widehat Y_b=f_\theta(X_{cb},M_{xcb},Y_{cb},M_{ycb},X_{qb},M_{xqb})$.
3. Average $L_t=B^{-1}\sum_b L_{\mathrm{episode},b}$.
4. Backpropagate $g_t=\nabla_\theta L_t$, accumulating sequentially with $L_b/B$.
5. Clip once: $g_{\mathrm{clip}}=g_t\min(1,\mathrm{GRAD\_CLIP\_NORM}/\lVert g_t\rVert_2)$.
6. AdamW updates moments:
   $$m_t=\beta_1m_{t-1}+(1-\beta_1)g_{\mathrm{clip}},\quad v_t=\beta_2v_{t-1}+(1-\beta_2)g_{\mathrm{clip}}^2,$$
   $$\widehat m_t=m_t/(1-\beta_1^t),\qquad\widehat v_t=v_t/(1-\beta_2^t).$$
7. AdamW updates parameters:
   $$\theta_t=(1-\eta\,\mathrm{WEIGHT\_DECAY})\theta_{t-1}
   -\eta\widehat m_t/(\sqrt{\widehat v_t}+\mathrm{ADAM\_EPS}).$$
8. Validate periodically and at the final update; save the best loss and apply patience.

Autograd, `clip_grad_norm_`, and `AdamW` perform steps 4-7. Logs report the mean episode loss since the previous log and elapsed seconds. The call below starts training when this cell runs, including during Run All.

Validation also reports unweighted standardized speed MSE and direction-component MSE separately. The scheduler halves the learning rate after three consecutive unsuccessful checks (PyTorch patience=2), before early stopping at ten. Combined losses are not directly comparable with earlier runs using a different direction weight.


In [ ]:
def train_model(model, optimizer, data, rng, device, sampling_settings, validation_episodes,
                max_steps, episodes_per_update, grad_clip_norm, log_interval,
                validation_interval, patience, min_delta, speed_weight, direction_weight,
                checkpoint_path, architecture, feature_names, target_names,
                metadata_columns, normalization, site_roles, preprocessing, *,
                lr_reduction_factor, lr_scheduler_patience, min_learning_rate):
    if min(max_steps, episodes_per_update, log_interval, validation_interval, patience) < 1:
        raise ValueError("Training counts, intervals, and patience must be positive.")
    if min_delta < 0 or grad_clip_norm <= 0:
        raise ValueError("Require nonnegative MIN_DELTA and positive gradient clipping norm.")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=lr_reduction_factor, patience=lr_scheduler_patience,
        threshold=min_delta, threshold_mode='abs', min_lr=min_learning_rate)
    history = {'training': [], 'validation': []}
    best_loss, patience_loss, stale = float('inf'), float('inf'), 0
    started, log_losses = perf_counter(), []
    log_speed_losses, log_direction_losses = [], []
    model.train()
    for step in range(1, max_steps+1):
        optimizer.zero_grad(set_to_none=True)
        update_loss = update_speed_loss = update_direction_loss = 0.0
        # 1. Sample fresh episodes, processed sequentially without padding.
        for _ in range(episodes_per_update):
            tensors, _ = sample_episode_tensors(data, rng, device, **sampling_settings)
            # 2. Predict without supplying query labels.
            prediction = predict_episode(model, tensors)
            # 3. Average the episode objective across B episodes.
            loss, speed_loss, direction_loss = masked_episode_loss(
                prediction, tensors['query_y'], tensors['query_y_mask'],
                speed_weight, direction_weight, return_components=True)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite training loss at step {step}.")
            update_loss += loss.item()/episodes_per_update
            update_speed_loss += speed_loss.item()/episodes_per_update
            update_direction_loss += direction_loss.item()/episodes_per_update
            # 4. Accumulate the gradient of the mean loss.
            (loss/episodes_per_update).backward()
        # 5. Clip the accumulated gradient once.
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm, error_if_nonfinite=True)
        # 6-7. Update AdamW moments and parameters once.
        optimizer.step()
        history['training'].append(dict(step=step, loss=update_loss, speed_loss=update_speed_loss,
                                        direction_loss=update_direction_loss, lr=optimizer.param_groups[0]['lr']))
        log_losses.append(update_loss)
        log_speed_losses.append(update_speed_loss)
        log_direction_losses.append(update_direction_loss)
        if step % log_interval == 0 or step == max_steps:
            print(f"Step {step}/{max_steps} ({100*step/max_steps:.1f}%) | "
                  f"mean training loss {np.mean(log_losses):.6f} | "
                  f"speed MSE {np.mean(log_speed_losses):.6f} | "
                  f"direction MSE {np.mean(log_direction_losses):.6f} | "
                  f"LR {optimizer.param_groups[0]['lr']:.2g} | elapsed {perf_counter()-started:.1f}s")
            log_losses.clear()
            log_speed_losses.clear()
            log_direction_losses.clear()
        # 8. Fixed validation episodes; restore training mode afterward.
        if step % validation_interval == 0 or step == max_steps:
            val_loss, val_speed, val_direction = validate_model(
                model, validation_episodes, device, speed_weight, direction_weight, return_components=True)
            history['validation'].append(dict(step=step, loss=val_loss, speed_loss=val_speed,
                                             direction_loss=val_direction, lr=optimizer.param_groups[0]['lr']))
            print(f"Validation step {step}: mean episode loss {val_loss:.6f} | "
                  f"speed MSE {val_speed:.6f} | direction MSE {val_direction:.6f}")
            previous_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_loss)
            if optimizer.param_groups[0]['lr'] < previous_lr:
                print(f"Learning rate reduced to {optimizer.param_groups[0]['lr']:.2g}")
            if val_loss < best_loss:
                best_loss = val_loss
                save_checkpoint(checkpoint_path, model, architecture, feature_names, target_names,
                                metadata_columns, normalization, site_roles, preprocessing, step, val_loss)
            if val_loss < patience_loss-min_delta:
                patience_loss, stale = val_loss, 0
            else:
                stale += 1
            if stale >= patience:
                print(f"Early stopping at step {step}; best validation loss {best_loss:.6f}")
                break
    return history


history = train_model(
    model, optimizer, prepared_data, rng, device, sampling_settings, validation_episodes,
    MAX_TRAINING_STEPS, EPISODES_PER_UPDATE, GRAD_CLIP_NORM, LOG_INTERVAL,
    VALIDATION_INTERVAL, EARLY_STOPPING_PATIENCE, EARLY_STOPPING_MIN_DELTA,
    SPEED_LOSS_WEIGHT, DIRECTION_LOSS_WEIGHT, CHECKPOINT_PATH, architecture,
    FEATURE_NAMES, TARGET_NAMES, prepared_data['metadata_columns'],
    prepared_data['normalization'], site_roles, preprocessing,
    lr_reduction_factor=LR_REDUCTION_FACTOR, lr_scheduler_patience=LR_SCHEDULER_PATIENCE,
    min_learning_rate=MIN_LEARNING_RATE)


## Full-period reconstruction

Choose `RECONSTRUCTION_CONTEXT_MODE="trained"` to sample up to the checkpoint's saved `n_context` usable campaign hours, or `"full"` to use every usable campaign hour (the default). Trained-count selection uses a separate supplied NumPy RNG, samples without replacement, and preserves chronological order. Coverage is checked on the full campaign before selection. Encode context once, then predict every requested UTC endpoint in chunks, retaining missing predictor masks and row order. The returned DataFrame includes Boolean `predictors_complete`: True only when every query predictor mask is observed (reference wind, reference/target ERA5, and retained metadata). Missing direction also makes this flag False, even though training allows missing direction. This conservative input-completeness flag is not a confidence score and does not inspect target observations or require query labels. All requested hours are returned, including incomplete ones. Metadata records the context mode, usable campaign hours, and selected count. Its loader reads target wind only within the campaign and applies saved normalization to all years, independent of the fitting cutoff.

$$\widehat s_{\mathrm{m/s}}=\max(0,\widehat z\,\sigma_{\mathrm{train}}+\mu_{\mathrm{train}}),\qquad
\widehat\theta=\operatorname{degrees}(\operatorname{atan2}(\widehat{\sin},\widehat{\cos}))\bmod360.$$

Direction is undefined for calm predicted speed or a numerically zero direction vector. Full campaign self-attention has quadratic context memory cost; query chunk size controls query memory only.


In [ ]:
# Full-period example after training; target may also be a test site for final evaluation.
# loaded_model, saved, saved_normalization = load_checkpoint(CHECKPOINT_PATH, device)
# arrays, details = assemble_reconstruction_inputs(
#     WIND_PATH, ERA5_PATH, METADATA_PATH, saved, saved_normalization,
#     target=saved['site_roles']['validation_sites'][0],
#     references=saved['site_roles']['development_sites'][:saved['preprocessing']['n_references']],
#     campaign_start_local=saved['preprocessing']['validation_campaign_start_dates'][0],
#     campaign_months=saved['preprocessing']['campaign_months'][0],
#     start_local=START_LOCAL, end_local=END_LOCAL,
#     context_mode=RECONSTRUCTION_CONTEXT_MODE, rng=np.random.default_rng(RANDOM_SEED))
# reconstructed, reconstruction_metadata = reconstruct_wind(
#     loaded_model, **arrays, normalization=saved_normalization, device=device,
#     query_chunk_size=RECONSTRUCTION_QUERY_CHUNK, reconstruction_metadata=details,
#     calm_speed_mps=saved['preprocessing']['calm_speed_mps'],
#     direction_vector_eps=saved['preprocessing']['direction_vector_eps'])


## Small smoke check

Use an independent sampler RNG, temporary model/optimizer, and temporary checkpoint. Restore PyTorch RNG state afterward, including CUDA, so the real training state is untouched. One real episode supplies tiny context/query subsets for attention checks. Verify finite loss/gradients, a parameter update, checkpoint agreement, and query chunk invariance. This does not train the real model.


In [ ]:
def run_smoke_check(data, architecture, device, sampling_settings, seed, optimizer_settings,
                    speed_weight, direction_weight, grad_clip_norm, feature_names, target_names,
                    metadata_columns, normalization, site_roles, preprocessing):
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices):
        torch.manual_seed(seed)
        episode = sample_training_episode(data, np.random.default_rng(seed), **sampling_settings)
        for prefix in ('context', 'query'):
            # Match the current helper's exact reference-slot and metadata layout.
            ids = np.searchsorted(data['times_utc'], episode[prefix+'_times_utc'])
            assembled_x, assembled_mask = assemble_predictors(
                data, episode['target_site_id'], episode['reference_site_ids'], ids,
                feature_names, metadata_columns)
            np.testing.assert_array_equal(assembled_x, episode[prefix+'_x'])
            np.testing.assert_array_equal(assembled_mask, episode[prefix+'_x_mask'])
            x, y = episode[prefix+'_x'], episode[prefix+'_y']
            xm, ym = episode[prefix+'_x_mask'], episode[prefix+'_y_mask']
            assert x.shape == xm.shape and y.shape == ym.shape
            assert x.shape[1] == len(feature_names) and y.shape[1] == len(target_names) == 3
            assert x.dtype == y.dtype == np.float32 and xm.dtype == ym.dtype == bool
            assert np.isfinite(x).all() and np.isfinite(y).all()
            assert (x[~xm] == 0).all() and (y[~ym] == 0).all()
            for kind in ('x', 'y', 'x_mask', 'y_mask', 'times_utc'):
                episode[prefix+'_'+kind] = episode[prefix+'_'+kind][:8].copy()
        tensors, details = episode_to_tensors(episode, device)
        temporary_model = WindContextModel(**architecture).to(device)
        temporary_optimizer = torch.optim.AdamW(temporary_model.parameters(), **optimizer_settings)
        before = next(temporary_model.parameters()).detach().clone()
        prediction = predict_episode(temporary_model, tensors)
        assert prediction.shape == tensors['query_y'].shape
        loss = masked_episode_loss(prediction, tensors['query_y'], tensors['query_y_mask'], speed_weight, direction_weight)
        assert torch.isfinite(loss)
        loss.backward()
        gradients = [p.grad for p in temporary_model.parameters() if p.grad is not None]
        assert gradients and all(torch.isfinite(g).all() for g in gradients)
        nn.utils.clip_grad_norm_(temporary_model.parameters(), grad_clip_norm, error_if_nonfinite=True)
        temporary_optimizer.step()
        assert not torch.equal(before, next(temporary_model.parameters()).detach())
        # Missing direction contributes zero; missing speed must fail.
        no_direction = tensors['query_y_mask'].clone()
        no_direction[..., 1:] = False
        speed_only = masked_episode_loss(prediction.detach(), tensors['query_y'], no_direction, speed_weight, direction_weight)
        expected = speed_weight*(prediction.detach()[..., 0]-tensors['query_y'][..., 0]).square().mean()
        torch.testing.assert_close(speed_only, expected)
        try:
            masked_episode_loss(prediction.detach(), tensors['query_y'], torch.zeros_like(no_direction), speed_weight, direction_weight)
        except ValueError:
            pass
        else:
            raise AssertionError("Missing speed supervision must fail.")
        temporary_model.eval()
        with torch.no_grad():
            expected_prediction = predict_episode(temporary_model, tensors)
        with TemporaryDirectory() as folder:
            path = Path(folder)/'smoke.pt'
            save_checkpoint(path, temporary_model, architecture, feature_names, target_names, metadata_columns,
                            normalization, site_roles, preprocessing, 1, loss.item())
            reloaded, saved, restored = load_checkpoint(path, device)
            with torch.no_grad():
                torch.testing.assert_close(predict_episode(reloaded, tensors), expected_prediction, rtol=1e-5, atol=1e-6)
                encoded = reloaded.encode_context(tensors['context_x'], tensors['context_x_mask'], tensors['context_y'], tensors['context_y_mask'])
                chunks = torch.cat([reloaded.predict_queries(encoded, tensors['query_x'][:, i:i+1], tensors['query_x_mask'][:, i:i+1])
                                    for i in range(tensors['query_x'].shape[1])], dim=1)
                torch.testing.assert_close(chunks, expected_prediction, rtol=1e-4, atol=1e-5)
            inputs = {k: episode[k] for k in ('context_x', 'context_x_mask', 'context_y', 'context_y_mask', 'query_x', 'query_x_mask', 'query_times_utc')}
            args = dict(normalization=restored, device=device, reconstruction_metadata=details,
                        calm_speed_mps=preprocessing['calm_speed_mps'], direction_vector_eps=preprocessing['direction_vector_eps'])
            one, _ = reconstruct_wind(reloaded, **inputs, query_chunk_size=1, **args)
            whole, _ = reconstruct_wind(reloaded, **inputs, query_chunk_size=len(episode['query_x']), **args)
            np.testing.assert_allclose(one.predicted_speed_mps, whole.predicted_speed_mps, rtol=1e-4, atol=1e-4)
            np.testing.assert_array_equal(one.predictors_complete, episode['query_x_mask'].all(axis=1))
            np.testing.assert_array_equal(one.predictors_complete, whole.predictors_complete)
            # Circular comparison handles angles near 0/360 degrees.
            a, b = one.predicted_direction_deg.to_numpy(), whole.predicted_direction_deg.to_numpy()
            np.testing.assert_array_equal(np.isnan(a), np.isnan(b))
            valid = np.isfinite(a) & np.isfinite(b)
            np.testing.assert_allclose((a[valid]-b[valid]+180) % 360 - 180, 0., atol=1e-2)
            assert str(one.time_utc.dt.tz) == 'UTC' and len(one) == len(episode['query_x'])
    print("Smoke check passed: real shapes/masks, finite loss/gradients, parameter update, checkpoint round trip, query chunks.")


run_smoke_check(
    prepared_data, architecture, device, sampling_settings, RANDOM_SEED,
    dict(lr=LEARNING_RATE, betas=ADAM_BETAS, eps=ADAM_EPS, weight_decay=WEIGHT_DECAY),
    SPEED_LOSS_WEIGHT, DIRECTION_LOSS_WEIGHT, GRAD_CLIP_NORM, FEATURE_NAMES, TARGET_NAMES,
    prepared_data['metadata_columns'], prepared_data['normalization'], site_roles, preprocessing)
